## Parameters

In [ ]:
dbutils.widgets.text("model_catalog", "")
dbutils.widgets.text("model_schema", "")
dbutils.widgets.text("logistic_model_name", "")
dbutils.widgets.text("lightgbm_model_name", "")
dbutils.widgets.text("pytorch_model_name", "")
dbutils.widgets.text("endpoint_name", "")

model_catalog = dbutils.widgets.get("model_catalog").strip()
model_schema = dbutils.widgets.get("model_schema").strip()
logistic_model_name = dbutils.widgets.get("logistic_model_name").strip()
lightgbm_model_name = dbutils.widgets.get("lightgbm_model_name").strip()
pytorch_model_name = dbutils.widgets.get("pytorch_model_name").strip()
endpoint_name = dbutils.widgets.get("endpoint_name").strip()

params = {
    "model_catalog": model_catalog,
    "model_schema": model_schema,
    "logistic_model_name": logistic_model_name,
    "lightgbm_model_name": lightgbm_model_name,
    "pytorch_model_name": pytorch_model_name,
    "endpoint_name": endpoint_name,
}
if not all(params.values()):
    missing = ", ".join(name for name, value in params.items() if not value)
    raise ValueError(f"All parameters must be non-empty; missing: {missing}")

logistic_path = f"{model_catalog}.{model_schema}.{logistic_model_name}"
lightgbm_path = f"{model_catalog}.{model_schema}.{lightgbm_model_name}"
pytorch_path = f"{model_catalog}.{model_schema}.{pytorch_model_name}"

print(f"Endpoint: {endpoint_name}")
print(f"Logistic: {logistic_path}")
print(f"LightGBM: {lightgbm_path}")
print(f"PyTorch:  {pytorch_path}")

## Resolve latest model versions

In [ ]:
import mlflow
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()


def latest_version(model_name: str) -> int:
    versions = client.search_model_versions(filter_string=f"name='{model_name}'")
    if not versions:
        raise ValueError(f"No registered versions found for {model_name}")
    return max(int(v.version) for v in versions)


versions = {
    "logistic_regression": latest_version(logistic_path),
    "lightgbm": latest_version(lightgbm_path),
    "pytorch_mlp": latest_version(pytorch_path),
}

for name, version in versions.items():
    print(f"{name}: version {version}")

## Build endpoint config

In [ ]:
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    Route,
    ServedEntityInput,
    ServingModelWorkloadType,
    TrafficConfig,
)

served_entities = [
    ServedEntityInput(
        name="logistic_regression",
        entity_name=logistic_path,
        entity_version=str(versions["logistic_regression"]),
        workload_type=ServingModelWorkloadType.CPU,
        workload_size="Small",
        scale_to_zero_enabled=True,
    ),
    ServedEntityInput(
        name="lightgbm",
        entity_name=lightgbm_path,
        entity_version=str(versions["lightgbm"]),
        workload_type=ServingModelWorkloadType.CPU,
        workload_size="Small",
        scale_to_zero_enabled=True,
    ),
    ServedEntityInput(
        name="pytorch_mlp",
        entity_name=pytorch_path,
        entity_version=str(versions["pytorch_mlp"]),
        workload_type=ServingModelWorkloadType.CPU,
        workload_size="Small",
        scale_to_zero_enabled=True,
    ),
]

traffic_config = TrafficConfig(
    routes=[
        Route(served_entity_name="logistic_regression", traffic_percentage=33),
        Route(served_entity_name="lightgbm", traffic_percentage=33),
        Route(served_entity_name="pytorch_mlp", traffic_percentage=34),
    ]
)

config = EndpointCoreConfigInput(
    name=endpoint_name,
    served_entities=served_entities,
    traffic_config=traffic_config,
)

print("Traffic split: 33% / 33% / 34%")
print("Scale-to-zero: enabled (required by workspace)")


## Create or update endpoint

In [ ]:
import time
from datetime import timedelta

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import OperationFailed
from databricks.sdk.errors.platform import ResourceDoesNotExist
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    EndpointStateConfigUpdate,
    EndpointStateReady,
)

w = WorkspaceClient()
DEPLOY_TIMEOUT = timedelta(minutes=30)
POLL_INTERVAL_SECONDS = 5


def is_recoverable_create_failure(exc: BaseException) -> bool:
    message = str(exc).lower()
    return "update_failed" in message or "quota exceeded" in message


def endpoint_has_quota_failure(name: str) -> bool:
    try:
        ep = w.serving_endpoints.get(name=name)
    except ResourceDoesNotExist:
        return False
    active_config = ep.pending_config or ep.config
    if active_config is None:
        return False
    for entity in active_config.served_entities or []:
        state = entity.state
        message = (state.deployment_state_message or "").lower() if state else ""
        if "quota exceeded" in message:
            return True
    return False


def wait_for_endpoint_deleted(name: str, timeout: timedelta = timedelta(minutes=2)) -> None:
    deadline = time.time() + timeout.total_seconds()
    while time.time() < deadline:
        try:
            w.serving_endpoints.get(name=name)
        except ResourceDoesNotExist:
            return
        time.sleep(POLL_INTERVAL_SECONDS)
    raise RuntimeError(f"Endpoint {name} still exists after delete")


def wait_for_endpoint_ready(name: str, timeout: timedelta = DEPLOY_TIMEOUT):
    deadline = time.time() + timeout.total_seconds()
    while time.time() < deadline:
        ep = w.serving_endpoints.get(name=name)
        config_update = ep.state.config_update if ep.state else None
        ready = ep.state.ready if ep.state else None

        if config_update == EndpointStateConfigUpdate.UPDATE_FAILED:
            raise RuntimeError(
                f"Endpoint {name} is in UPDATE_FAILED. Delete it in the Serving UI "
                f"(or `databricks serving-endpoints delete {name}`), recreate it there, "
                "then re-run this job to verify."
            )
        if (
            ready == EndpointStateReady.READY
            and config_update == EndpointStateConfigUpdate.NOT_UPDATING
        ):
            return ep
        time.sleep(POLL_INTERVAL_SECONDS)
    raise RuntimeError(f"Timed out waiting for endpoint {name} to become READY")


def rollout_entities_incrementally(name: str) -> None:
    """Add served entities one at a time to avoid free-tier quota spikes."""
    rollout_steps = [
        (served_entities[:1], [Route(served_entity_name=served_entities[0].name, traffic_percentage=100)]),
        (
            served_entities[:2],
            [
                Route(served_entity_name=served_entities[0].name, traffic_percentage=50),
                Route(served_entity_name=served_entities[1].name, traffic_percentage=50),
            ],
        ),
        (served_entities, traffic_config.routes),
    ]

    first_entities, first_routes = rollout_steps[0]
    first_config = EndpointCoreConfigInput(
        name=name,
        served_entities=first_entities,
        traffic_config=TrafficConfig(routes=first_routes),
    )
    print(f"Creating endpoint with first model: {first_entities[0].name}")
    w.serving_endpoints.create_and_wait(
        name=name,
        config=first_config,
        timeout=DEPLOY_TIMEOUT,
    )

    for step_entities, step_routes in rollout_steps[1:]:
        print(f"Adding models: {[entity.name for entity in step_entities]}")
        w.serving_endpoints.update_config_and_wait(
            name=name,
            served_entities=step_entities,
            traffic_config=TrafficConfig(routes=step_routes),
            timeout=DEPLOY_TIMEOUT,
        )


def create_endpoint(name: str) -> None:
    """Create all models in one shot (~3x faster than incremental rollout)."""
    print(f"Creating endpoint with all models: {name}")
    try:
        w.serving_endpoints.create_and_wait(
            name=name,
            config=config,
            timeout=DEPLOY_TIMEOUT,
        )
    except OperationFailed as exc:
        if not (is_recoverable_create_failure(exc) or endpoint_has_quota_failure(name)):
            raise
        print("Bulk create failed; falling back to incremental rollout")
        try:
            failed = w.serving_endpoints.get(name=name)
            if (
                failed.state is not None
                and (
                    failed.state.config_update == EndpointStateConfigUpdate.UPDATE_FAILED
                    or failed.state.ready != EndpointStateReady.READY
                )
            ):
                w.serving_endpoints.delete(name=name)
                wait_for_endpoint_deleted(name)
        except ResourceDoesNotExist:
            pass
        rollout_entities_incrementally(name)


existing = None
try:
    existing = w.serving_endpoints.get(name=endpoint_name)
except ResourceDoesNotExist:
    pass

desired_model_versions = {
    entity.entity_name: entity.entity_version for entity in served_entities
}

if existing is not None:
    config_update = (
        existing.state.config_update if existing.state is not None else None
    )
    is_ready = (
        existing.state is not None
        and existing.state.ready == EndpointStateReady.READY
        and config_update == EndpointStateConfigUpdate.NOT_UPDATING
    )

    if config_update == EndpointStateConfigUpdate.UPDATE_FAILED:
        if endpoint_has_quota_failure(endpoint_name):
            print(f"Recovering UPDATE_FAILED endpoint via incremental rollout: {endpoint_name}")
            w.serving_endpoints.delete(name=endpoint_name)
            wait_for_endpoint_deleted(endpoint_name)
            rollout_entities_incrementally(endpoint_name)
            endpoint = w.serving_endpoints.get(name=endpoint_name)
        else:
            raise RuntimeError(
                f"Endpoint {endpoint_name} is UPDATE_FAILED. Delete it in the Serving UI "
                f"(or `databricks serving-endpoints delete {endpoint_name}`), then re-run this job."
            )
    elif not is_ready:
        print(f"Endpoint provisioning in progress, waiting: {endpoint_name}")
        endpoint = wait_for_endpoint_ready(endpoint_name)
    else:
        deployed_model_versions = {
            entity.entity_name: entity.entity_version
            for entity in (existing.config.served_entities or [])
        }
        if deployed_model_versions == desired_model_versions:
            print(f"Endpoint already READY with latest versions: {endpoint_name}")
            endpoint = existing
        else:
            print(f"Updating versions on READY endpoint: {endpoint_name}")
            endpoint = w.serving_endpoints.update_config_and_wait(
                name=endpoint_name,
                served_entities=served_entities,
                traffic_config=traffic_config,
                timeout=DEPLOY_TIMEOUT,
            )
else:
    create_endpoint(endpoint_name)
    endpoint = w.serving_endpoints.get(name=endpoint_name)

print(f"Endpoint {endpoint.name} ready state: {endpoint.state.ready}")


## Verify READY

In [ ]:
from databricks.sdk.service.serving import EndpointStateReady

endpoint = w.serving_endpoints.get(name=endpoint_name)

if endpoint.state is None or endpoint.state.ready != EndpointStateReady.READY:
    deployment_details = []
    active_config = endpoint.pending_config or endpoint.config
    if active_config is not None:
        for entity in active_config.served_entities or []:
            state = entity.state
            deployment_details.append(
                {
                    "name": entity.name,
                    "deployment": None if state is None else str(state.deployment),
                    "message": None if state is None else state.deployment_state_message,
                }
            )
    quota_errors = [
        d["message"]
        for d in deployment_details
        if d.get("message") and "quota exceeded" in d["message"].lower()
    ]
    if quota_errors:
        raise RuntimeError(
            "Serving quota exceeded for 3 scale-to-zero models on this workspace. "
            "Delete failed endpoints and other custom serving endpoints to free capacity, "
            f"or upgrade the workspace. Details: {quota_errors[0]}"
        )
    raise RuntimeError(
        f"Endpoint not READY: {endpoint.state}; deployment details: {deployment_details}"
    )

print(f"Endpoint {endpoint.name} is READY")
print("Served entities:")
for entity in endpoint.config.served_entities:
    print(f"  {entity.name}: {entity.entity_name} v{entity.entity_version}")

print("Traffic:")
for route in endpoint.config.traffic_config.routes:
    route_name = route.served_entity_name or route.served_model_name
    print(f"  {route_name}: {route.traffic_percentage}%")
